## In- and out-of-phase form stress (amplitude) at each layer

In [1]:
%matplotlib inline

import datetime, time
import numpy as np
import xesmf as xe
import xarray as xr
import netCDF4 as nc
import cmocean as cm
import matplotlib.ticker
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from pathlib import Path
from scipy.optimize import curve_fit
from matplotlib.colors import LogNorm

import warnings
warnings.filterwarnings('ignore')

### construct in- and out-of-phase form stress

In [2]:
data_path = Path("/g/data/nm03/lxy581/evaluate/amp_phase")
data_stress = data_path / "form_stress_amp_pha_test_013.nc"
data_vel_u = data_path / "form_u_amp_pha_test_013.nc"
data_vel_v = data_path / "form_v_amp_pha_test_013.nc"

stress = xr.open_dataset(data_stress)
vel_u  = xr.open_dataset(data_vel_u)
vel_v  = xr.open_dataset(data_vel_v)

In [3]:
out_path = Path("/scratch/nm03/lxy581/mom6/archive/tides_008_global_sigma_SAL_2layer_x02/output014")
out_grid = out_path / "ocean_static.nc"

static = xr.open_dataset(out_grid)

In [4]:
stress_amp_x = stress.amp_x
stress_pha_x = stress.phase_x
u_pha = vel_u.phase_x

In [5]:
stress_amp_y = stress.amp_y
stress_pha_y = stress.phase_y
v_pha = vel_v.phase_y

#### x-dir, in

In [6]:
x_in = stress_amp_x * np.cos(stress_pha_x - u_pha)

In [ ]:
x_in.plot(vmin=-6e+1,vmax=6e+1,cmap="RdBu_r")

#### x-dir, out

In [ ]:
x_out = stress_amp_x * np.sin(stress_pha_x - u_pha)

In [ ]:
x_out.plot(vmin=-6e+1,vmax=6e+1,cmap="RdBu_r")

#### y-dir, in

In [ ]:
y_in = stress_amp_y * np.cos(stress_pha_y - v_pha)

#### y-dir, out

In [ ]:
y_out = stress_amp_y * np.sin(stress_pha_y - v_pha)

### Interpolate to a depth, identify on which side of topography these red & blue dots lie 

In [ ]:
dep_path = Path("/g/data/nm03/lxy581/evaluate/amp_phase")
dep_file = dep_path / "depth_h2u_8km.nc"

dep_interp = xr.open_dataset(dep_file)
depth_u = dep_interp.depthu
depth_v = dep_interp.depthv

In [ ]:
depth_u.plot(vmin=0,vmax=6500,cmap="jet")

In [ ]:
mask_depth_u = (depth_u > 3000) & (depth_u < 3500)
# mask_depth_u.plot(vmin=0,vmax=2,cmap="RdBu_r")

In [ ]:
mask_depth_v = (depth_v > 3000) & (depth_v < 3500)
# mask_depth_v.plot(vmin=0,vmax=2,cmap="RdBu_r")

In [ ]:
# choose values in this depth range
x_in_cut = x_in * mask_depth_u
x_out_cut = x_out * mask_depth_u
y_in_cut = y_in * mask_depth_v
y_out_cut = y_out * mask_depth_v

#### Know which side of topography these points are at
If left side: -1
If right side: 1

In [ ]:
x_in_cut.yh.isel(yh=600)

In [ ]:
x_in_cut_slice = x_in_cut.isel(yh=600).values
x_in_cut_slice

In [ ]:
sides_u = 0 * depth_u

# diff_u = depth_u.diff(dim="xq")
diff_u = depth_u.roll(xq=-1, roll_coords=False) - depth_u

# left of topog: -1, right of topog: 1
sides_u = xr.where(diff_u > 0, 1.0, -1.0)

In [ ]:
print(diff_u.shape)
print(sides_u.shape)

### Blue: left of topog  | Red: right of topog

In [ ]:
sides_u.plot(vmin=-1.5,vmax=1.5,cmap="RdBu_r")

In [ ]:
sides_u_masked = sides_u * mask_depth_u
# sides_u_masked.plot(vmin=-1.5,vmax=1.5,cmap="RdBu_r")

In [ ]:
x_in_cut_slice = x_in_cut.isel(yh=600).values
x_in_cut_slice

In [ ]:
sides_u_slice = sides_u_masked.isel(yh=600).values
mask_left = (sides_u_slice == -1) & (x_in_cut_slice != 0)
ind_left = np.where(mask_left)[0]
ind_left

In [ ]:
x_in_cut_slice[ind_left[:]]

In [ ]:
mask_right = (sides_u_slice == 1) & (x_in_cut_slice != 0)
ind_right = np.where(mask_right)[0]
ind_right

In [ ]:
x_in_cut_slice[ind_right[:]]

In [ ]:
fig = plt.figure(figsize=(20, 8))
xq = depth_u.xq.values
dep_slice = -depth_u.isel(yh=600).values
plt.plot(xq,dep_slice,'k',alpha=0.4)
plt.xlim(np.nanmin(xq),np.nanmax(xq))
plt.tick_params(labelsize=20)
plt.ylabel('Depth (m)',fontsize=20)
plt.xlabel('Lon',fontsize=20)
plt.plot((xq[0],xq[-1]),(-3000,-3000),'k--',alpha=0.5)
plt.plot((xq[0],xq[-1]),(-3500,-3500),'k--',alpha=0.5)
plt.scatter(xq[ind_left],dep_slice[ind_left],marker='o',s=6, c='blue')
plt.scatter(xq[ind_right],dep_slice[ind_right],marker='o',s=6, c='red')
plt.savefig('/g/data/nm03/lxy581/evaluate/amp_phase/select_points_across_topog.png', dpi=600, bbox_inches='tight')